# Разведочный анализ данных EMNIST Balanced

**Магистерская диссертация. Глава 2, § 2.4**

Этот ноутбук вычисляет статистики и строит визуализации, необходимые для написания § 2.4 «Разведочный анализ данных» диссертационной работы. Все артефакты (PNG-графики и числовые значения) сохраняются в `/content/eda_outputs/` для последующего скачивания и включения в текст работы.

**Содержание:**
1. Загрузка EMNIST Balanced (с корректировкой транспонирования)
2. Проверка сбалансированности классов в train / test
3. Статистика яркости пикселей (Mean, Std, гистограмма)
4. Доля «активных» пикселей
5. Средние изображения по 47 классам
6. Примеры (сэмплы) по 47 классам
7. Визуальное сопоставление близких классов (O / 0, I / 1 / L, S / 5, g / 9 / q)
8. Итоговая сводка чисел для § 2.4

**Примечание о работе с EMNIST.** Изображения в EMNIST хранятся в транспонированном виде относительно MNIST (повёрнуты на 90° и зеркально отражены). В ноутбуке это исправляется на этапе загрузки операцией `tensor.transpose(1, 2)`.


## 1. Подготовка окружения

In [ ]:
# В Colab torch и torchvision уже установлены; на всякий случай — тихая проверка
import sys, subprocess
try:
    import torch, torchvision
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchvision'])
    import torch, torchvision

print('torch       :', torch.__version__)
print('torchvision :', torchvision.__version__)
print('CUDA доступна:', torch.cuda.is_available())


In [ ]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from torchvision import datasets, transforms
import torch

# --- настройки matplotlib для академического стиля ---
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 100,
    'savefig.dpi': 200,
    'savefig.bbox': 'tight',
    'savefig.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
})

OUT_DIR = '/content/eda_outputs'
os.makedirs(OUT_DIR, exist_ok=True)

# Контейнер для итоговых числовых результатов
STATS = {}

print('Окружение готово. Артефакты будут сохраняться в:', OUT_DIR)


## 2. Загрузка EMNIST Balanced

Скачивание происходит штатными средствами `torchvision.datasets.EMNIST`. Транспонирование исправляется лямбдой в pipeline.


In [ ]:
DATA_DIR = '/content/emnist_data'

# Корректирующий transpose: EMNIST хранится транспонированным относительно MNIST
transform = transforms.Compose([
    transforms.ToTensor(),                    # [0, 1], shape (1, 28, 28)
    transforms.Lambda(lambda t: t.transpose(1, 2)),  # исправляем ориентацию
])

train_set = datasets.EMNIST(
    root=DATA_DIR, split='balanced', train=True,
    download=True, transform=transform,
)
test_set = datasets.EMNIST(
    root=DATA_DIR, split='balanced', train=False,
    download=True, transform=transform,
)

print(f'Train: {len(train_set):>6d} образцов')
print(f'Test : {len(test_set):>6d} образцов')
print(f'Всего: {len(train_set) + len(test_set):>6d}')

sample_img, sample_label = train_set[0]
print(f'Форма одного изображения: {tuple(sample_img.shape)}')
print(f'Диапазон значений: [{sample_img.min():.3f}, {sample_img.max():.3f}]')
print(f'Тип данных: {sample_img.dtype}')


In [ ]:
# Полное соответствие индекс -> символ для EMNIST Balanced
LABEL_TO_CHAR = {
    0:'0',1:'1',2:'2',3:'3',4:'4',5:'5',6:'6',7:'7',8:'8',9:'9',
    10:'A',11:'B',12:'C',13:'D',14:'E',15:'F',16:'G',17:'H',18:'I',19:'J',
    20:'K',21:'L',22:'M',23:'N',24:'O',25:'P',26:'Q',27:'R',28:'S',29:'T',
    30:'U',31:'V',32:'W',33:'X',34:'Y',35:'Z',
    36:'a',37:'b',38:'d',39:'e',40:'f',41:'g',42:'h',43:'n',44:'q',45:'r',46:'t'
}

assert len(LABEL_TO_CHAR) == 47, 'Должно быть ровно 47 классов'

# Метки в виде массивов numpy
train_labels = np.array(train_set.targets)
test_labels = np.array(test_set.targets)

print(f'Метки train: shape={train_labels.shape}, min={train_labels.min()}, max={train_labels.max()}')
print(f'Метки test : shape={test_labels.shape},  min={test_labels.min()}, max={test_labels.max()}')


## 3. Проверка сбалансированности классов

Для EMNIST Balanced ожидается ровно **2 400 train + 400 test образцов на каждый из 47 классов**. Проверим эмпирически.


In [ ]:
train_counts = Counter(train_labels.tolist())
test_counts  = Counter(test_labels.tolist())

train_per_class = np.array([train_counts[i] for i in range(47)])
test_per_class  = np.array([test_counts[i] for i in range(47)])

print('Train: образцов на класс — min={}, max={}, среднее={:.1f}, std={:.3f}'.format(
    train_per_class.min(), train_per_class.max(), train_per_class.mean(), train_per_class.std()))
print('Test : образцов на класс — min={}, max={}, среднее={:.1f}, std={:.3f}'.format(
    test_per_class.min(),  test_per_class.max(),  test_per_class.mean(),  test_per_class.std()))

balanced = (train_per_class.std() == 0) and (test_per_class.std() == 0)
print(f'\nНабор полностью сбалансирован: {balanced}')

STATS['train_total']        = int(train_per_class.sum())
STATS['test_total']         = int(test_per_class.sum())
STATS['train_per_class']    = int(train_per_class.mean())
STATS['test_per_class']     = int(test_per_class.mean())
STATS['fully_balanced']     = bool(balanced)
STATS['n_classes']          = 47


In [ ]:
# --- График: распределение образцов по 47 классам ---
fig, ax = plt.subplots(figsize=(14, 4.5))
x = np.arange(47)
width = 0.42

ax.bar(x - width/2, train_per_class, width, label='train', color='#3b6ea5')
ax.bar(x + width/2, test_per_class,  width, label='test',  color='#c44e52')

ax.set_xticks(x)
ax.set_xticklabels([LABEL_TO_CHAR[i] for i in range(47)], fontsize=9)
ax.set_xlabel('Класс')
ax.set_ylabel('Число образцов')
ax.set_title('Распределение образцов по 47 классам EMNIST Balanced')
ax.legend(loc='upper right', frameon=False)
ax.grid(axis='y', alpha=0.3)
ax.set_axisbelow(True)

# Подпись о точных значениях
ax.text(0.01, 0.97,
        f'train: ровно {train_per_class[0]} образцов / класс\ntest:  ровно {test_per_class[0]} образцов / класс',
        transform=ax.transAxes, va='top', ha='left',
        fontsize=9, bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='gray', alpha=0.9))

out_path = os.path.join(OUT_DIR, 'fig_2_4_1_class_distribution.png')
plt.savefig(out_path)
plt.show()
print('Сохранено:', out_path)


## 4. Статистика яркости пикселей

Вычисляем:
- среднее значение пикселя по всей обучающей выборке (нужно для нормализации в § 2.5),
- стандартное отклонение (тоже для нормализации),
- гистограмму распределения значений пикселей.

Для эффективности грузим данные пакетами через `DataLoader`.


In [ ]:
from torch.utils.data import DataLoader

loader = DataLoader(train_set, batch_size=512, shuffle=False, num_workers=2)

# --- среднее и std в один проход ---
n_pixels = 0
sum_px   = 0.0
sum_sq   = 0.0
# для гистограммы
hist_bins = 256
hist_counts = np.zeros(hist_bins, dtype=np.int64)

for batch, _ in loader:
    arr = batch.numpy().reshape(-1)
    n_pixels += arr.size
    sum_px   += arr.sum()
    sum_sq   += (arr ** 2).sum()
    # гистограмма по значениям [0, 1]
    counts, _ = np.histogram(arr, bins=hist_bins, range=(0.0, 1.0))
    hist_counts += counts

mean_px = sum_px / n_pixels
var_px  = sum_sq / n_pixels - mean_px ** 2
std_px  = float(np.sqrt(var_px))
mean_px = float(mean_px)

print(f'Всего пикселей в train: {n_pixels:,}'.replace(',', ' '))
print(f'Mean (на [0,1]):  {mean_px:.4f}')
print(f'Std  (на [0,1]):  {std_px:.4f}')
print(f'Mean (на [0,255]): {mean_px*255:.2f}')
print(f'Std  (на [0,255]): {std_px*255:.2f}')

STATS['pixel_mean_normalized'] = round(mean_px, 4)
STATS['pixel_std_normalized']  = round(std_px,  4)
STATS['pixel_mean_0_255']      = round(mean_px*255, 2)
STATS['pixel_std_0_255']       = round(std_px*255,  2)


In [ ]:
# --- График: гистограмма яркости (log-scale) ---
bin_centers = np.linspace(0, 1, hist_bins, endpoint=False) + 0.5/hist_bins

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# (a) линейная шкала
axes[0].bar(bin_centers, hist_counts, width=1.0/hist_bins, color='#3b6ea5', edgecolor='none')
axes[0].set_xlabel('Значение пикселя (нормализованное)')
axes[0].set_ylabel('Количество пикселей')
axes[0].set_title('(а) Распределение яркости пикселей, линейная шкала')
axes[0].set_xlim(0, 1)

# (б) логарифмическая шкала (более информативна для EMNIST)
axes[1].bar(bin_centers, hist_counts, width=1.0/hist_bins, color='#c44e52', edgecolor='none')
axes[1].set_yscale('log')
axes[1].set_xlabel('Значение пикселя (нормализованное)')
axes[1].set_ylabel('Количество пикселей (log)')
axes[1].set_title('(б) То же, логарифмическая шкала')
axes[1].set_xlim(0, 1)

# вертикальные линии mean ± std
for ax in axes:
    ax.axvline(mean_px,            color='black', ls='--', lw=1, alpha=0.7)
    ax.axvline(mean_px + std_px,   color='black', ls=':',  lw=1, alpha=0.5)
    ax.axvline(mean_px - std_px,   color='black', ls=':',  lw=1, alpha=0.5)

fig.suptitle(f'Распределение значений пикселей в обучающей выборке (Mean={mean_px:.4f}, Std={std_px:.4f})', y=1.02)
out_path = os.path.join(OUT_DIR, 'fig_2_4_2_pixel_histogram.png')
plt.savefig(out_path)
plt.show()
print('Сохранено:', out_path)


## 5. Доля «активных» пикселей

«Активный» пиксель — тот, чьё значение превышает порог `threshold`. Для EMNIST (белый штрих на чёрном фоне) обычно используют порог 0,1 — он отделяет штрих символа от фона. Этот показатель характеризует «плотность» рисунка символа.


In [ ]:
THRESHOLD = 0.1

n_active = 0
n_total  = 0
for batch, _ in loader:
    arr = batch.numpy()
    n_active += int((arr > THRESHOLD).sum())
    n_total  += arr.size

active_ratio = n_active / n_total
print(f'Порог активности: {THRESHOLD}')
print(f'Доля активных пикселей: {active_ratio*100:.2f} %')
print(f'(в среднем {active_ratio*28*28:.1f} активных пикселей из 784 на изображение)')

STATS['active_pixel_threshold'] = THRESHOLD
STATS['active_pixel_ratio']     = round(active_ratio, 4)
STATS['active_pixels_per_image'] = round(active_ratio*28*28, 1)


## 6. Средние изображения по классам

Для каждого из 47 классов строим попиксельное среднее всех его 2 400 обучающих образцов. Это даёт «канонический» вид каждого символа.


In [ ]:
# Накапливаем суммы по классам в один проход
class_sums   = np.zeros((47, 28, 28), dtype=np.float64)
class_counts = np.zeros(47,            dtype=np.int64)

for batch, labels in loader:
    arr = batch.numpy().squeeze(1)  # (B, 28, 28)
    lbls = labels.numpy()
    for c in range(47):
        mask = (lbls == c)
        if mask.any():
            class_sums[c]   += arr[mask].sum(axis=0)
            class_counts[c] += int(mask.sum())

mean_images = class_sums / class_counts[:, None, None]
print('Средние изображения посчитаны, shape:', mean_images.shape)


In [ ]:
# --- График: сетка 47 средних изображений ---
fig, axes = plt.subplots(6, 8, figsize=(11, 8.5))

for c in range(48):
    ax = axes[c // 8, c % 8]
    if c < 47:
        ax.imshow(mean_images[c], cmap='gray_r', vmin=0, vmax=1)
        ax.set_title(LABEL_TO_CHAR[c], fontsize=11, pad=2)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.grid(False)

# скрываем последнюю пустую клетку
axes[5, 7].axis('off')

fig.suptitle('Средние изображения по 47 классам EMNIST Balanced (по 2 400 образцов каждое)',
             fontsize=12, y=1.0)
plt.tight_layout()
out_path = os.path.join(OUT_DIR, 'fig_2_4_3_mean_images.png')
plt.savefig(out_path)
plt.show()
print('Сохранено:', out_path)


## 7. Примеры (сэмплы) по классам

По одному случайному образцу на каждый из 47 классов — для иллюстрации разнообразия почерков.


In [ ]:
rng = np.random.RandomState(42)

# Индексы первых вхождений каждого класса (для удобства возьмём случайные)
samples = {}
for c in range(47):
    cls_indices = np.where(train_labels == c)[0]
    samples[c] = int(rng.choice(cls_indices))

# --- сетка 6×8 ---
fig, axes = plt.subplots(6, 8, figsize=(11, 8.5))
for c in range(48):
    ax = axes[c // 8, c % 8]
    if c < 47:
        img, _ = train_set[samples[c]]
        ax.imshow(img.squeeze().numpy(), cmap='gray_r', vmin=0, vmax=1)
        ax.set_title(LABEL_TO_CHAR[c], fontsize=11, pad=2)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.grid(False)

axes[5, 7].axis('off')

fig.suptitle('Случайные примеры из 47 классов EMNIST Balanced', fontsize=12, y=1.0)
plt.tight_layout()
out_path = os.path.join(OUT_DIR, 'fig_2_4_4_samples.png')
plt.savefig(out_path)
plt.show()
print('Сохранено:', out_path)


## 8. Сопоставление визуально близких классов

Эти группы — «жёсткое ядро» задачи EMNIST, формирующее основные ошибки классификации (см. § 4.10 и § 2.3).


In [ ]:
confusable_groups = [
    ('0 vs O',      [0, 24]),         # цифра 0 и буква O
    ('1 vs I vs L', [1, 18, 21]),     # цифра 1, буква I, буква L
    ('5 vs S',      [5, 28]),         # цифра 5 и буква S
    ('9 vs g vs q', [9, 41, 44]),     # цифра 9, буква g, буква q
]

fig, axes = plt.subplots(len(confusable_groups), 4, figsize=(9, 9))

for row, (title, idxs) in enumerate(confusable_groups):
    for col in range(4):
        ax = axes[row, col]
        if col < len(idxs):
            c = idxs[col]
            ax.imshow(mean_images[c], cmap='gray_r', vmin=0, vmax=1)
            ax.set_title(f"«{LABEL_TO_CHAR[c]}» (среднее)", fontsize=10, pad=2)
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.grid(False)
    axes[row, 0].set_ylabel(title, fontsize=11, rotation=0, labelpad=55, ha='right', va='center')

fig.suptitle('Средние изображения для визуально близких классов EMNIST Balanced',
             fontsize=12, y=1.0)
plt.tight_layout()
out_path = os.path.join(OUT_DIR, 'fig_2_4_5_confusable_pairs.png')
plt.savefig(out_path)
plt.show()
print('Сохранено:', out_path)


## 9. Итоговая сводка для § 2.4

Все числовые значения, которые понадобятся при написании текста параграфа.


In [ ]:
print('=' * 60)
print('  ИТОГОВЫЕ ЧИСЛА ДЛЯ § 2.4')
print('=' * 60)
for k, v in STATS.items():
    print(f'  {k:<32s} = {v}')
print('=' * 60)

# сохраняем в JSON, чтобы можно было скопировать
stats_path = os.path.join(OUT_DIR, 'eda_stats.json')
with open(stats_path, 'w', encoding='utf-8') as f:
    json.dump(STATS, f, ensure_ascii=False, indent=2)
print('JSON-файл сохранён:', stats_path)


## 10. Скачивание артефактов

Список всех созданных файлов и их скачивание одним архивом.


In [ ]:
# Архивируем папку с артефактами
import shutil
zip_path = '/content/eda_outputs.zip'
shutil.make_archive('/content/eda_outputs', 'zip', OUT_DIR)

print('Файлы в', OUT_DIR + ':')
for f in sorted(os.listdir(OUT_DIR)):
    full = os.path.join(OUT_DIR, f)
    size_kb = os.path.getsize(full) / 1024
    print(f'  {f:<40s}  {size_kb:>7.1f} KB')

print(f'\nАрхив: {zip_path}')

# Авто-скачивание в Colab
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('Скачивание автоматическое доступно только в Colab.')
